Step 1 — Install dependencies & set your API key

In [ ]:
import os, getpass
ANTHROPIC_API_KEY = ""

MODEL = 'claude-sonnet-4-6'  # fast + cheap for a workshop; opus-4-8 for harder reasoning

     

Step 2 — A tiny FastAPI backend (the 'external API')

In [3]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',  'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',          'qty': 2, 'status': 'processing','total': 58.0},
    'A1003': {'id': 'A1003', 'item': '4K monitor',         'qty': 1, 'status': 'delivered', 'total': 410.0},
}

@app.get('/orders/{order_id}')
def get_order(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

client = TestClient(app)
print(client.get('/orders/A1001').json())
     

{'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped', 'total': 129.0}


Step 3 — A Redis memory layer

In [8]:
import json, time, fakeredis

class RedisMemory:
    def __init__(self, r, session_id: str, history_limit: int = 40):
        self.r = r
        self.sid = session_id
        self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'
        self.f_key = f'facts:{session_id}'

    # ---- short-term: conversation turns ----
    def append_turn(self, role: str, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)  # keep only the tail

    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

    # ---- long-term: durable facts ----
    def set_fact(self, key: str, value: str, ttl_seconds: int | None = None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds:
            self.r.expire(self.f_key, ttl_seconds)

    def get_fact(self, key: str):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v

    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

r = fakeredis.FakeStrictRedis()
mem = RedisMemory(r, session_id='demo-user')
mem.set_fact('name', 'Asha')
mem.append_turn('user', 'hello')
print('facts:', mem.all_facts())
print('history:', mem.load_history())

facts: {'name': 'Asha'}
history: [{'role': 'user', 'content': 'hello'}]


Step 4 — Declare the tools (JSON schemas)

In [9]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by its ID and return item, quantity, status and total.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'remember_fact',
        'description': 'Persist a durable fact about the user (e.g. shipping preference) for future turns.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'key':   {'type': 'string', 'description': 'Short fact key, e.g. "shipping_pref".'},
                'value': {'type': 'string', 'description': 'The fact value to store.'},
            },
            'required': ['key', 'value'],
        },
    },
    {
        'name': 'recall_fact',
        'description': 'Retrieve a previously stored fact about the user by key. Returns empty if unknown.',
        'input_schema': {
            'type': 'object',
            'properties': {'key': {'type': 'string', 'description': 'The fact key to look up.'}},
            'required': ['key'],
        },
    },
]
print(len(TOOLS), 'tools declared')
     

3 tools declared


Step 5 — A dispatch map: tool name → Python function

In [10]:
def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()

def tool_remember_fact(key: str, value: str):
    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}

DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

print(run_tool('get_order', {'order_id': 'A1002'}))
print(run_tool('get_order', {'order_id': 'A9999'}))

({'error': 'TypeError("SyncAPIClient.get() missing 1 required keyword-only argument: \'cast_to\'")'}, True)
({'error': 'TypeError("SyncAPIClient.get() missing 1 required keyword-only argument: \'cast_to\'")'}, True)


Step 6 — The agent loop (driven by stop_reason)

In [ ]:
import json

SYSTEM = (
    'You are an order-support assistant. Use get_order for any order question. '
    'Use remember_fact / recall_fact to keep durable user preferences across turns. '
    'Be concise.'
)

def agent_turn(user_text, max_steps=6, verbose=True):
    mem.append_turn('user', user_text)
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose: print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic(api_key="")
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            # echo the assistant turn (text + tool_use blocks) back into the transcript
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue
        # end_turn
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

print(agent_turn('What is the status of order A1002?'))

  → tool: get_order({'order_id': 'A1002'})
I'm sorry, it looks like I encountered a technical error while trying to retrieve order **A1002**. Here's what I'd suggest:

1. **Try again in a moment** — it may be a temporary issue.
2. **Contact support directly** if the problem persists.

Would you like me to try looking it up again, or is there anything else I can help you with?


In [20]:

print(agent_turn('Please remember that my shipping preference is express.'))
print('---')
print(agent_turn('What did I say my shipping preference was?'))
print('---')
print('Raw facts in Redis:', mem.all_facts())
print('History length:', len(mem.load_history()), 'turns')

  → tool: remember_fact({'key': 'shipping_pref', 'value': 'express'})
Got it! I've saved your shipping preference as **express**. I'll keep that in mind for future interactions. Is there anything else I can help you with?
---
  → tool: recall_fact({'key': 'shipping_pref'})
Your shipping preference is set to **express**. Is there anything else I can help you with?
---
Raw facts in Redis: {'name': 'Asha', 'shipping_pref': 'express'}
History length: 12 turns


In [21]:
#Step 8 — Multi-turn chat demo
for msg in [
    'Hi, I am Asha.',
    'How much was order A1003?',
    'Remember that my budget cap is 500 dollars.',
    'Given my budget cap, was that order within it?',
]:
    print('USER:', msg)
    print('AGENT:', agent_turn(msg, verbose=False))
    print()
     

USER: Hi, I am Asha.
AGENT: Got it, Asha! I'll remember that. How can I help you today?

USER: How much was order A1003?
AGENT: I'm sorry, Asha — I encountered a technical error while trying to retrieve order **A1003**. Please try again in a moment, or contact support directly if the issue persists. Is there anything else I can help you with?

USER: Remember that my budget cap is 500 dollars.
AGENT: Done, Asha! I've saved your budget cap as **$500**. Is there anything else I can help you with?

USER: Given my budget cap, was that order within it?
AGENT: I was able to confirm your budget cap is **$500**, but unfortunately I ran into a technical error retrieving order **A1003** again. I'm unable to compare the order total at this time.

Could you try again in a moment? Or if you know the order total, I can quickly tell you whether it falls within your $500 budget cap!



Extention tasks

task 1

If chat history becomes too long, don’t keep sending all old messages to the LLM.
Instead, take the old messages, summarize them into one short summary message, store that summary, and keep only recent turns in full.

This is called rolling summary / conversation compaction.

In [ ]:
import os
from anthropic import Anthropic

os.environ["ANTHROPIC_API_KEY"] = ""
clientA = Anthropic()

In [51]:
import json

SYSTEM = (
    "You are an order-support assistant.\n"
    "- Use get_order ONLY when the user explicitly asks about an order or provides an order ID.\n"
    "- Do NOT call get_order for greetings, chit-chat, or general questions.\n"
    "- Use remember_fact only when the user explicitly asks you to remember/store a preference or fact.\n"
    "- Use recall_fact only when needed to answer a question about previously stored user facts.\n"
    "- Be concise."
)

def maybe_compact_history(mem, model, max_turns=12, keep_last=6, verbose=True):
    """
    If chat history grows beyond max_turns, summarize the oldest turns using Claude
    and replace them with one synthetic assistant summary message.
    """
    history = mem.load_history()

    # If history is small, do nothing
    if len(history) <= max_turns:
        return

    # Split into old turns and recent turns
    old_turns = history[:-keep_last]
    recent_turns = history[-keep_last:]

    if verbose:
        print(f'… compacting history: {len(history)} turns -> summarize first {len(old_turns)} turns')

    # If LIVE=False, use a simple mock summary
    if not LIVE:
        summary_text = 'Earlier conversation included order questions and stored user preferences.'
    else:
        from anthropic import Anthropic
        clientA = Anthropic()

        # Convert old turns to plain text for summarization
        convo_text = "\n".join(
            f"{m['role']}: {m['content']}" for m in old_turns
        )

        summary_prompt = (
            "Summarize the following earlier conversation for future context.\n"
            "Keep only durable facts, preferences, past order IDs discussed, and unresolved issues.\n"
            "Do NOT phrase it like a current user request.\n"
            "Do NOT ask to call tools.\n"
            "Use short bullet points.\n\n"
            f"{convo_text}"
        )

        resp = clientA.messages.create(
            model=model,
            max_tokens=300,
            system="You summarize chat history for memory compaction.",
            messages=[{'role': 'user', 'content': summary_prompt}]
        )

        summary_text = ''.join(
            b.text for b in resp.content if b.type == 'text'
        ).strip()

    # Build one synthetic summary message
    summary_msg = {
        'role': 'assistant',
        'content': f'[Summary of earlier conversation]\n{summary_text}'
    }

    # Replace old history with compacted history
    new_history = [summary_msg] + recent_turns

    # overwrite redis history
    mem.r.delete(mem.h_key)
    for item in new_history:
        mem.r.rpush(mem.h_key, json.dumps(item))

    if verbose:
        print('… history compacted successfully')


def agent_turn(user_text, max_steps=6, verbose=True):
    # 1) store user turn
    mem.append_turn('user', user_text)

    # 2) compact history if too long
    maybe_compact_history(mem, MODEL, max_turns=12, keep_last=6, verbose=verbose)

    # 3) load final history after possible compaction
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose:
            print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    print("ok")
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )

        if resp.stop_reason == 'tool_use':
            # Store assistant tool request in local transcript
            messages.append({
                'role': 'assistant',
                'content': [b.model_dump() for b in resp.content]
            })

            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose:
                        print(f'  → tool: {block.name}({block.input})')

                    out, is_err = run_tool(block.name, block.input)

                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })

            messages.append({'role': 'user', 'content': results})
            continue

        # normal final assistant text
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text

    return '(stopped: max steps reached)'

In [48]:
maybe_compact_history(mem, MODEL, max_turns=12, keep_last=6, verbose=True)

… compacting history: 15 turns -> summarize first 9 turns
… history compacted successfully


In [52]:
def show_history():
    print("\n----- CURRENT HISTORY -----")
    for i, msg in enumerate(mem.load_history(), 1):
        print(f"{i}. {msg}")
    print("Total turns:", len(mem.load_history()))

In [53]:
tests = [
    "Hi",
    "My name is Asha",
    "Remember that I prefer express shipping",
    "My city is Bangalore",
    "What is the status of order A1001?",

]

for t in tests:
    print("\nUSER:", t)
    print("BOT :", agent_turn(t))
    show_history()


USER: Hi
… compacting history: 13 turns -> summarize first 7 turns
… history compacted successfully
ok
BOT : Hi Syed! 👋 How can I help you today?

----- CURRENT HISTORY -----
1. {'role': 'assistant', 'content': "[Summary of earlier conversation]\nHere's the updated summary of the conversation:\n\n- **User:** Syed, located in **Bangalore**\n- **Email:** syed@example.com\n- **Shipping preference:** Express shipping\n- **Payment preference:** UPI\n- **Orders inquired:** A1001, A1002, and A1003 — all could not be retrieved due to technical issues (unresolved)"}
2. {'role': 'assistant', 'content': "Done! I've saved your email as **syed@example.com**. Is there anything else I can help you with?"}
3. {'role': 'user', 'content': 'What order did I ask about earlier?'}
4. {'role': 'assistant', 'content': 'You asked about the following orders earlier:\n\n- **A1001**\n- **A1002**\n- **A1003**\n\nIs there anything else I can help you with, Syed?'}
5. {'role': 'user', 'content': 'Summarize what you

In [54]:
maybe_compact_history(mem, MODEL, max_turns=12, keep_last=6, verbose=True)

In [55]:
show_history()


----- CURRENT HISTORY -----
1. {'role': 'assistant', 'content': '[Summary of earlier conversation]\nHi Syed! Welcome back! 😊 How can I help you today?'}
2. {'role': 'assistant', 'content': 'Hi Syed! 👋 How can I help you today?'}
3. {'role': 'user', 'content': 'My name is Asha'}
4. {'role': 'assistant', 'content': 'Got it, Asha! Would you like me to remember your name for future reference?'}
5. {'role': 'user', 'content': 'Remember that I prefer express shipping'}
6. {'role': 'assistant', 'content': "Done! I've remembered that you prefer **Express shipping**. Is there anything else I can help you with, Asha?"}
7. {'role': 'user', 'content': 'My city is Bangalore'}
8. {'role': 'assistant', 'content': "Got it! Should I remember that your city is **Bangalore**? Just confirm and I'll save it for you!"}
9. {'role': 'user', 'content': 'What is the status of order A1001?'}
10. {'role': 'assistant', 'content': "✅ I've saved your city (**Bangalore**) and shipping preference (**Express shipping*

Extension task 2

In [56]:
import re

EMAIL_RE = re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b')
CARD_RE  = re.compile(r'\b(?:\d[ -]*?){13,19}\b')

def looks_like_email(text: str) -> bool:
    return bool(EMAIL_RE.search(text or ""))

def looks_like_card_number(text: str) -> bool:
    if not text:
        return False
    m = CARD_RE.search(text)
    if not m:
        return False
    digits = re.sub(r'\D', '', m.group(0))
    return 13 <= len(digits) <= 19


def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()


def tool_remember_fact(key: str, value: str):
    # Reject email-like values
    if looks_like_email(value):
        return {
            'error': (
                "Refusing to store sensitive data in memory: the value looks like an email address. "
                "Please store a non-sensitive preference instead."
            )
        }

    # Reject card-number-like values
    if looks_like_card_number(value):
        return {
            'error': (
                "Refusing to store sensitive data in memory: the value looks like a payment card number. "
                "Please do not store card details in memory."
            )
        }

    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}


def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}


DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}


def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

In [57]:
agent_turn("Remember my email is syed@example.com")

ok
  → tool: remember_fact({'key': 'city', 'value': 'Bangalore'})
  → tool: get_order({'order_id': 'A1001'})
  → tool: remember_fact({'key': 'email', 'value': 'syed@example.com'})


"Here's a summary:\n\n- 🏙️ **City (Bangalore):** Saved successfully!\n- 📦 **Order A1001:** Unfortunately, I'm still unable to retrieve the order details due to a technical issue. Please try again later or contact support.\n- 📧 **Email (syed@example.com):** I'm unable to store your email address as the system doesn't allow saving sensitive data like email addresses for security reasons.\n\nIs there anything else I can help you with, Asha?"

In [59]:
tool_remember_fact(key="email", value="syed@example.com")

{'error': 'Refusing to store sensitive data in memory: the value looks like an email address. Please store a non-sensitive preference instead.'}